# DynaICL Stage 0.6: noisy-observation probe (Google Colab, CPU only)

Runs the **frozen** Stage 0.6 code (`docs/PREREG_stage06.md`) from the frozen snapshot `stage06_frozen_snapshot.zip`.

* Snapshot sha256: `4114fa43cf737c643c341cccd4026d6fcb54b460bfb12ac4a8a2de8352b492e5`
* Manifest sha256: `461dfec8144104f4d35a513dbf1411041494876331f8f969dda4a9cea09be0c0`. Compare it with the value **you** stored outside the workspace.

**Sections**
* Sections 1-3: setup, environment qualification and a manifest check. Safe to run at any time.
* This notebook contains NO cell that launches a run. The Phase D pilot has its own notebook
  (`colab/pilot/DynaICL_Stage06_PhaseD_PILOT.ipynb`); the confirmatory run has none (removed before Phase D).

In [ ]:
!python3 --version
!nproc

## 1. Mount Drive and unpack the frozen snapshot

In [ ]:
import hashlib, os, zipfile
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/dynaicl_stage06", exist_ok=True)
SNAPSHOT_ON_DRIVE = ""          # optional: e.g. "/content/drive/MyDrive/stage06_frozen_snapshot.zip"
EXPECTED_SNAPSHOT = "4114fa43cf737c643c341cccd4026d6fcb54b460bfb12ac4a8a2de8352b492e5"
MANIFEST_SHA = "461dfec8144104f4d35a513dbf1411041494876331f8f969dda4a9cea09be0c0"      # must equal the value you stored outside the workspace
if SNAPSHOT_ON_DRIVE:
    src = SNAPSHOT_ON_DRIVE
else:
    from google.colab import files
    src = next(iter(files.upload()))       # pick stage06_frozen_snapshot.zip
sha = hashlib.sha256(open(src, "rb").read()).hexdigest()
assert sha == EXPECTED_SNAPSHOT, f"snapshot sha256 mismatch: {sha}"
os.makedirs("/content/DynaICL", exist_ok=True)
zipfile.ZipFile(src).extractall("/content/DynaICL")
os.chdir("/content/DynaICL")
os.environ["MANIFEST_SHA"] = MANIFEST_SHA
print("snapshot OK; manifest file sha256:", hashlib.sha256(open("docs/FREEZE_MANIFEST_STAGE06.json", "rb").read()).hexdigest())

## 2. Install the pinned stack (pip warnings about other Colab packages are harmless)

In [ ]:
!pip install -q -r /content/DynaICL/requirements.txt
!python3 -c "import numpy, scipy, yaml; print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| pyyaml', yaml.__version__)"

## 3. Environment qualification (a few minutes)

If this environment differs from the frozen environment manifest (Python version, packages or platform), the runner
refuses unless this qualification passes here: V4 (full Tier-1), V6 (quadrature convergence), V11 (parity null) and
V13 (batch = per-context). The report is written to Drive, outside the frozen file set.

In [ ]:
%cd /content/DynaICL
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python3 -m dynaicl.experiments.stage06_probe qualify --config config/stage06.yaml --manifest docs/FREEZE_MANIFEST_STAGE06.json --expected-manifest-sha256 $MANIFEST_SHA --out /content/drive/MyDrive/dynaicl_stage06/qualification.json
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python3 -m dynaicl.experiments.stage06_probe qualify --config config/stage06_pilot.yaml --manifest docs/FREEZE_MANIFEST_STAGE06.json --expected-manifest-sha256 $MANIFEST_SHA --out /content/drive/MyDrive/dynaicl_stage06/qualification_pilot.json
!python3 -m dynaicl.experiments.stage06_probe verify --config config/stage06.yaml --manifest docs/FREEZE_MANIFEST_STAGE06.json --expected-manifest-sha256 $MANIFEST_SHA --qualification /content/drive/MyDrive/dynaicl_stage06/qualification.json

## 4. Archive a finished run and download it

In [ ]:
import glob, os, tarfile
runs = sorted(glob.glob("/content/drive/MyDrive/dynaicl_stage06/runs/*") + glob.glob("/content/drive/MyDrive/dynaicl_stage06/pilot_runs/*"), key=os.path.getmtime)
run = runs[-1]
print(run)
archive = "/content/" + os.path.basename(run) + ".tar.gz"
with tarfile.open(archive, "w:gz") as tf:
    tf.add(run, arcname=os.path.basename(run))
os.system(f"cp {archive} /content/drive/MyDrive/dynaicl_stage06/")
from google.colab import files
files.download(archive)